# Data Analysis: Energy Efficiency Guidelines Evaluation
This notebook processes raw energy consumption data collected from 30 runs of baseline and optimized configurations across multiple guidelines. 

### Statistical Pipeline:
1. **Normality Test**: Shapiro-Wilk (Expected: Non-normal, $p < 0.05$)
2. **Hypothesis Testing**: Aligned Rank Transform ANOVA (ART-ANOVA) to compare medians.
3. **Holm-Bonferroni correction**: Correction applied to ensure that the false-positive rate remains at 5%.
4. **Effect Size**: Cliff's Delta to determine the magnitude of the impact.

In [8]:
import os
import glob
import re
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [9]:
import os
import glob
import re
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols


def run_fast_cliffs_delta(lst1, lst2):
    """Fast, vectorized implementation of Cliff's Delta using numpy."""
    arr1 = np.array(lst1)
    arr2 = np.array(lst2)
    diff = arr1[:, None] - arr2
    return np.mean(np.sign(diff))


def run_normality_test(data):
    """Shapiro-Wilk normality test."""
    if len(data) < 3:
        return 1.0
    stat, p_value = stats.shapiro(data)
    return p_value


def extract_run_energy(csv_path):
    """
    Correct extraction: energy consumed during ONE run =
    last value of PACKAGE_ENERGY (J) minus the first value
    (this is a monotonically increasing counter).
    Returns None if the column is missing, there's not enough
    data, or the delta looks like a rollover artifact.
    """
    try:
        df = pd.read_csv(csv_path)
        energy_col = [c for c in df.columns if 'PACKAGE_ENERGY' in c.upper()]
        if not energy_col:
            return None
        series = pd.to_numeric(df[energy_col[0]], errors='coerce').dropna()
        if len(series) < 2:
            return None
        delta = series.iloc[-1] - series.iloc[0]
        # Filter out rollovers / clearly invalid deltas
        if delta <= 0:
            return None
        return float(delta)
    except Exception:
        return None


DATA_DIR = r"C:\Users\ruxin\OneDrive\Documente\thesis_exp\final_data"
all_results = []

if not os.path.exists(DATA_DIR):
    print(f"Error: Cannot find '{DATA_DIR}' folder. Please make sure the notebook is saved right next to it.")
else:
    print(f"Located '{DATA_DIR}'. Starting structured folder data extraction...")

    for bench_name in sorted(os.listdir(DATA_DIR)):
        bench_path = os.path.join(DATA_DIR, bench_name)
        if not os.path.isdir(bench_path):
            continue

        sub_folders = os.listdir(bench_path)

        base_sub = [f for f in sub_folders if "baseline" in f.lower() and os.path.isdir(os.path.join(bench_path, f))]
        if not base_sub:
            print(f"Skipping {bench_name}: No baseline subfolder found.")
            continue

        base_folder_path = os.path.join(bench_path, base_sub[0])
        baseline_files = glob.glob(os.path.join(base_folder_path, "*.csv"))

        base_runs = []
        skipped_base = 0
        for f in baseline_files:
            val = extract_run_energy(f)
            if val is not None:
                base_runs.append(val)
            else:
                skipped_base += 1

        if len(base_runs) == 0:
            print(f"Skipping {bench_name}: Could not extract valid energy data from baseline CSVs.")
            continue

        g_subs = [f for f in sub_folders if "baseline" not in f.lower() and os.path.isdir(os.path.join(bench_path, f))]

        for g_folder in g_subs:
            g_match = re.search(r"_(g\d+)", g_folder, re.IGNORECASE)
            guideline_label = g_match.group(1).lower() if g_match else g_folder

            opt_folder_path = os.path.join(bench_path, g_folder)
            optimized_files = glob.glob(os.path.join(opt_folder_path, "*.csv"))

            opt_runs = []
            skipped_opt = 0
            for f in optimized_files:
                val = extract_run_energy(f)
                if val is not None:
                    opt_runs.append(val)
                else:
                    skipped_opt += 1

            if len(opt_runs) == 0:
                continue

            if skipped_base or skipped_opt:
                print(f"  [{bench_name} / {guideline_label}] skipped {skipped_base} baseline "
                      f"and {skipped_opt} optimized runs (invalid/rollover deltas)")

            p_shapiro_base = run_normality_test(base_runs)
            p_shapiro_opt = run_normality_test(opt_runs)

            df_base = pd.DataFrame({'Energy': base_runs, 'Group': 'baseline'})
            df_opt = pd.DataFrame({'Energy': opt_runs, 'Group': 'optimized'})
            df_combined = pd.concat([df_base, df_opt]).reset_index(drop=True)
            df_combined['Ranked_Energy'] = stats.rankdata(df_combined['Energy'])

            model = ols('Ranked_Energy ~ Group', data=df_combined).fit()
            p_anova = sm.stats.anova_lm(model, typ=2)['PR(>F)'].iloc[0]

            median_base = np.median(base_runs)
            median_optimized = np.median(opt_runs)

            if p_anova < 0.05:
                conclusion = "Saves Energy" if median_optimized < median_base else "Wastes Energy"
            else:
                conclusion = "No Statistically Significant Impact"

            cliffs_d = run_fast_cliffs_delta(opt_runs, base_runs)

            all_results.append({
                'Guideline': guideline_label,
                'Benchmark': bench_name,
                'N_Base': len(base_runs),
                'N_Opt': len(opt_runs),
                'Shapiro_Base_p': round(p_shapiro_base, 5),
                'Shapiro_Opt_p': round(p_shapiro_opt, 5),
                'ART_ANOVA_p': round(p_anova, 5),
                'Median_Baseline_J': round(median_base, 4),
                'Median_Optimized_J': round(median_optimized, 4),
                'Conclusion': conclusion,
                'Cliffs_Delta': round(cliffs_d, 4)
            })

    print("--- Pipeline processing executed successfully! ---")

summary_df = pd.DataFrame(all_results)

# Multiple comparisons correction (Holm-Bonferroni), applied globally
if not summary_df.empty:
    from statsmodels.stats.multitest import multipletests
    reject, p_corrected, _, _ = multipletests(summary_df['ART_ANOVA_p'], alpha=0.05, method='holm')
    summary_df['ART_ANOVA_p_holm'] = np.round(p_corrected, 5)
    summary_df['Significant_after_correction'] = reject

    # Recompute conclusion using the corrected p-value
    def corrected_conclusion(row):
        if row['Significant_after_correction']:
            return "Saves Energy" if row['Median_Optimized_J'] < row['Median_Baseline_J'] else "Wastes Energy"
        return "No Statistically Significant Impact"

    summary_df['Conclusion_corrected'] = summary_df.apply(corrected_conclusion, axis=1)

    summary_df = summary_df.sort_values(by=['Guideline', 'Benchmark']).reset_index(drop=True)

summary_df


Located 'C:\Users\ruxin\OneDrive\Documente\thesis_exp\final_data'. Starting structured folder data extraction...
  [binary_trees / g1] skipped 1 baseline and 0 optimized runs (invalid/rollover deltas)
  [binary_trees / g18] skipped 1 baseline and 0 optimized runs (invalid/rollover deltas)
  [binary_trees / g19] skipped 1 baseline and 1 optimized runs (invalid/rollover deltas)
  [binary_trees / g3] skipped 1 baseline and 0 optimized runs (invalid/rollover deltas)
  [binary_trees / g7] skipped 1 baseline and 0 optimized runs (invalid/rollover deltas)
  [binary_trees / g8] skipped 1 baseline and 1 optimized runs (invalid/rollover deltas)
  [django / g3] skipped 0 baseline and 1 optimized runs (invalid/rollover deltas)
  [glg1-mndset / g25] skipped 1 baseline and 0 optimized runs (invalid/rollover deltas)
  [glg1-monte_carlo / g25] skipped 1 baseline and 0 optimized runs (invalid/rollover deltas)
  [results-fft / g18] skipped 0 baseline and 1 optimized runs (invalid/rollover deltas)
  [res

,Guideline,Benchmark,N_Base,N_Opt,Shapiro_Base_p,Shapiro_Opt_p,ART_ANOVA_p,Median_Baseline_J,Median_Optimized_J,Conclusion,Cliffs_Delta,ART_ANOVA_p_holm,Significant_after_correction,Conclusion_corrected
0,g1,binary_trees,29,30,0.00065,0.01660,0.00000,2945.6993,939.1821,Saves Energy,-1.0000,0.00000,True,Saves Energy
1,g1,nbody,30,30,0.00000,0.00032,0.00001,31.0222,26.6387,Saves Energy,-0.6111,0.00043,True,Saves Energy
2,g1,results-bubble-sort,30,30,0.00000,0.00000,0.00001,450.6334,491.2979,Wastes Energy,0.6222,0.00043,True,Wastes Energy
3,g1,results-fft,30,30,0.09090,0.29784,0.00000,3735.1506,3226.5308,Saves Energy,-1.0000,0.00000,True,Saves Energy
4,g1,results-lu,29,30,0.33714,0.31453,0.00000,6554.9112,1960.5226,Saves Energy,-1.0000,0.00000,True,Saves Energy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,g8,binary_trees,29,21,0.00065,0.01105,0.00000,2945.6993,639.4772,Saves Energy,-1.0000,0.00000,True,Saves Energy
131,g8,nbody,30,30,0.00000,0.02564,0.00000,31.0222,2.9695,Saves Energy,-1.0000,0.00000,True,Saves Energy
132,g8,results-bubble-sort,30,30,0.00000,0.00180,0.00000,450.6334,20.9039,Saves Energy,-1.0000,0.00000,True,Saves Energy
133,g8,results-sieves-ros,30,30,0.00000,0.00000,0.00000,683.6377,70.3191,Saves Energy,-1.0000,0.00000,True,Saves Energy


In [10]:
# Shapiro-Wilk normality table

shapiro_df = summary_df[['Guideline', 'Benchmark', 'Shapiro_Base_p', 'Shapiro_Opt_p']].copy()
shapiro_df['Normal_Base']     = shapiro_df['Shapiro_Base_p'] > 0.05
shapiro_df['Normal_Optimized'] = shapiro_df['Shapiro_Opt_p'] > 0.05

# Overall summary: how many of the 2*N groups (baseline + optimized) are normal?
n_groups = len(shapiro_df) * 2
n_normal = shapiro_df['Normal_Base'].sum() + shapiro_df['Normal_Optimized'].sum()
pct_normal = 100 * n_normal / n_groups
pct_nonnormal = 100 - pct_normal

print(f"Total groups tested: {n_groups}")
print(f"Normal (p > 0.05):     {n_normal}  ({pct_normal:.1f}%)")
print(f"Non-normal (p <= 0.05): {n_groups - n_normal}  ({pct_nonnormal:.1f}%)")
print()
print(f'"Shapiro-Wilk tests showed that {pct_nonnormal:.1f}% of the {n_groups} '
      f'baseline/optimized groups significantly deviated from normality (p <= 0.05), '
      f'confirming the use of a non-parametric ART-ANOVA approach."')
print()

# Export full per-pair table (appendix candidate)
shapiro_df.to_csv("shapiro_energy_results.csv", index=False)
print("Saved full table to shapiro_energy_results.csv")

shapiro_df


Total groups tested: 270
Normal (p > 0.05):     139  (51.5%)
Non-normal (p <= 0.05): 131  (48.5%)

"Shapiro-Wilk tests showed that 48.5% of the 270 baseline/optimized groups significantly deviated from normality (p <= 0.05), confirming the use of a non-parametric ART-ANOVA approach."

Saved full table to shapiro_energy_results.csv


,Guideline,Benchmark,Shapiro_Base_p,Shapiro_Opt_p,Normal_Base,Normal_Optimized
0,g1,binary_trees,0.00065,0.01660,False,False
1,g1,nbody,0.00000,0.00032,False,False
2,g1,results-bubble-sort,0.00000,0.00000,False,False
3,g1,results-fft,0.09090,0.29784,True,True
4,g1,results-lu,0.33714,0.31453,True,True
...,...,...,...,...,...,...
130,g8,binary_trees,0.00065,0.01105,False,False
131,g8,nbody,0.00000,0.02564,False,False
132,g8,results-bubble-sort,0.00000,0.00180,False,False
133,g8,results-sieves-ros,0.00000,0.00000,False,False


In [11]:
# Per-guideline summary table (uses the FULL summary_df, all benchmarks)

# Guideline -> Family mapping
GUIDELINE_FAMILY = {
    'g1': 'Code Optimization', 'g2': 'Code Optimization', 'g3': 'Code Optimization',
    'g4': 'Code Optimization', 'g5': 'Code Optimization', 'g6': 'Code Optimization',
    'g7': 'Code Optimization',
    'g8': 'Multithreading', 'g13': 'Multithreading', 'g25': 'Multithreading',
    'g15': 'Native Code', 'g17': 'Native Code', 'g24': 'Native Code', 'g27': 'Native Code',
    'g18': 'Function Calls', 'g19': 'Function Calls',
    'g23': 'Network',
    'g26': 'Other', 'g28': 'Other',
}

guideline_summary = []

for guideline, g_df in summary_df.groupby('Guideline'):
    n_total = len(g_df)
    n_saves = (g_df['Conclusion_corrected'] == 'Saves Energy').sum()
    n_wastes = (g_df['Conclusion_corrected'] == 'Wastes Energy').sum()
    n_noimpact = (g_df['Conclusion_corrected'] == 'No Statistically Significant Impact').sum()
    median_delta = g_df['Cliffs_Delta'].median()

    guideline_summary.append({
        'Guideline': guideline.upper(),
        'Family': GUIDELINE_FAMILY.get(guideline, 'Unknown'),
        'N_Benchmarks': n_total,
        'Saves': n_saves,
        'Wastes': n_wastes,
        'No_Impact': n_noimpact,
        'Median_Cliffs_Delta': round(median_delta, 3),
    })

guideline_summary_df = pd.DataFrame(guideline_summary)

guideline_summary_df['_sort'] = guideline_summary_df['Guideline'].str[1:].astype(int)
guideline_summary_df = guideline_summary_df.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)

print(guideline_summary_df.to_string(index=False))

# Generate LaTeX rows
def fmt_delta(d):
    return f"{d:+.2f}"  # show sign explicitly: -1.00 = saves, +1.00 = wastes

latex_lines = []
for _, row in guideline_summary_df.iterrows():
    line = (f"{row['Guideline']} & {row['Family']} & {row['N_Benchmarks']} & "
            f"{row['Saves']} & {row['Wastes']} & {row['No_Impact']} & "
            f"{fmt_delta(row['Median_Cliffs_Delta'])} \\\\ \\hline")
    latex_lines.append(line)

latex_table_body = "\n".join(latex_lines)
print("\n\n--- LaTeX rows ---\n")
print(latex_table_body)

with open("guideline_summary_energy_table_rows.tex", "w") as f:
    f.write(latex_table_body)
print("\n\nSaved to guideline_summary_energy_table_rows.tex")


Guideline            Family  N_Benchmarks  Saves  Wastes  No_Impact  Median_Cliffs_Delta
       G1 Code Optimization            11      5       3          3               -0.011
       G2 Code Optimization             5      4       0          1               -1.000
       G3 Code Optimization            13      5       1          7               -0.149
       G4 Code Optimization             6      3       0          3               -0.546
       G5 Code Optimization             5      3       1          1               -0.469
       G6 Code Optimization             5      3       0          2               -0.760
       G7 Code Optimization            11      2       1          8               -0.244
       G8    Multithreading             5      5       0          0               -1.000
      G13    Multithreading             5      3       2          0               -1.000
      G15       Native Code            10      6       4          0               -1.000
      G17       Nativ

In [12]:
# Per-family aggregation: which families decrease energy the most? 

GUIDELINE_FAMILY = {
    'g1': 'Code Optimization', 'g2': 'Code Optimization', 'g3': 'Code Optimization',
    'g4': 'Code Optimization', 'g5': 'Code Optimization', 'g6': 'Code Optimization',
    'g7': 'Code Optimization',
    'g8': 'Multithreading', 'g13': 'Multithreading', 'g25': 'Multithreading',
    'g15': 'Native Code', 'g17': 'Native Code', 'g24': 'Native Code', 'g27': 'Native Code',
    'g18': 'Function Calls', 'g19': 'Function Calls',
    'g23': 'Network',
    'g26': 'Other', 'g28': 'Other',
}

FAMILY_ORDER = ['Code Optimization', 'Multithreading', 'Native Code',
                'Function Calls', 'Network', 'Other']

summary_df['Family'] = summary_df['Guideline'].map(GUIDELINE_FAMILY).fillna('Unknown')

# 1. Per-family counts and median Cliff's Delta
family_rows = []
for family in FAMILY_ORDER:
    fam = summary_df[summary_df['Family'] == family]
    if fam.empty:
        continue
    n_total   = len(fam)
    n_saves   = (fam['Conclusion_corrected'] == 'Saves Energy').sum()
    n_wastes  = (fam['Conclusion_corrected'] == 'Wastes Energy').sum()
    n_no      = (fam['Conclusion_corrected'] == 'No Statistically Significant Impact').sum()
    med_delta = fam['Cliffs_Delta'].median()
    pct_saves = round(100 * n_saves / n_total, 1)

    family_rows.append({
        'Family':            family,
        'N_Comparisons':     n_total,
        'Saves':             n_saves,
        'Wastes':            n_wastes,
        'No_Impact':         n_no,
        'Saves_%':           pct_saves,
        'Median_Cliffs_d':   round(med_delta, 3),
    })

family_df = pd.DataFrame(family_rows)

# Sort by most energy-saving family (most negative median delta first)
family_df_sorted = family_df.sort_values('Median_Cliffs_d')

print("=" * 65)
print("PER-FAMILY SUMMARY — energy impact across all benchmarks")
print("=" * 65)
print(family_df_sorted.to_string(index=False))
print()
print("Negative Cliff's δ = guideline saves energy vs baseline.")
print("Ranking by median δ shows which family helps energy the most.\n")

# 2. Rank families from most to least beneficial
print("RANKING (most energy-saving → least):")
for rank, row in enumerate(family_df_sorted.itertuples(), 1):
    direction = "saves" if row.Median_Cliffs_d < 0 else "wastes"
    print(f"  {rank}. {row.Family:<22} | {row.Saves}/{row.N_Comparisons} saves "
          f"({row._6}%) | median δ = {row.Median_Cliffs_d:+.3f} → {direction} energy")

# 3. Export LaTeX rows 
latex_lines = []
for _, row in family_df_sorted.iterrows():
    line = (f"{row['Family']} & {row['N_Comparisons']} & "
            f"{row['Saves']} & {row['Wastes']} & {row['No_Impact']} & "
            f"{row['Saves_%']}\\% & {row['Median_Cliffs_d']:+.3f} \\\\ \\hline")
    latex_lines.append(line)

print("\n--- LaTeX rows for Discussion table ---")
print("\n".join(latex_lines))

family_df_sorted.to_csv("family_energy_summary_table.csv", index=False)
print("\nSaved to family_energy_summary_table.csv")


PER-FAMILY SUMMARY — energy impact across all benchmarks
           Family  N_Comparisons  Saves  Wastes  No_Impact  Saves_%  Median_Cliffs_d
   Multithreading             15     13       2          0     86.7           -1.000
      Native Code             32     18      11          3     56.2           -1.000
          Network              5      3       2          0     60.0           -1.000
Code Optimization             56     25       6         25     44.6           -0.289
   Function Calls             16      7       3          6     43.8           -0.160
            Other             11      5       6          0     45.5            0.944

Negative Cliff's δ = guideline saves energy vs baseline.
Ranking by median δ shows which family helps energy the most.

RANKING (most energy-saving → least):
  1. Multithreading         | 13/15 saves (86.7%) | median δ = -1.000 → saves energy
  2. Native Code            | 18/32 saves (56.2%) | median δ = -1.000 → saves energy
  3. Network       

In [13]:
# Select 5 representative benchmarks per guideline + generate LaTeX rows 

# 1. Find the "core" benchmarks: the ones tested across the most guidelines
benchmark_counts = summary_df.groupby('Benchmark')['Guideline'].nunique().sort_values(ascending=False)
print("Benchmarks ranked by how many guidelines they were tested with:")
print(benchmark_counts)
print()

core_benchmarks = benchmark_counts.head(5).index.tolist()
print(f"Proposed core set of 5 benchmarks (used for the main table): {core_benchmarks}")
print()

# 2. For each guideline, pick up to 5 benchmarks:
#    - prefer the core set (if that guideline was tested on them)
#    - fall back to whatever else is available, until 5 are reached
selected_rows = []

for guideline in sorted(summary_df['Guideline'].unique(),
                         key=lambda g: int(g[1:])): 
    g_df = summary_df[summary_df['Guideline'] == guideline]
    available = g_df['Benchmark'].tolist()

    # benchmarks from the core set that this guideline actually has
    chosen = [b for b in core_benchmarks if b in available]

    # fill remaining slots (up to 5) with other available benchmarks
    others = [b for b in available if b not in chosen]
    chosen += others[:max(0, 5 - len(chosen))]
    chosen = chosen[:5]

    for b in chosen:
        row = g_df[g_df['Benchmark'] == b].iloc[0]
        selected_rows.append(row)

table_df = pd.DataFrame(selected_rows)
print(f"\nSelected {len(table_df)} rows total ({len(table_df)//len(summary_df['Guideline'].unique())} avg per guideline)")
table_df[['Guideline', 'Benchmark', 'ART_ANOVA_p_holm', 'Median_Baseline_J', 'Median_Optimized_J', 'Conclusion_corrected']]


Benchmarks ranked by how many guidelines they were tested with:
Benchmark
results-lu                 12
results-sor                11
spectral_norm              11
results-monte_carlo        11
results-sieves-ros         10
results-fft                10
results-bubble-sort         9
results-sparse_mat_mult     9
nbody                       8
results-mandelbrot-set      7
binary_trees                6
django                      6
results-richards            5
glg1-sparse                 1
glg1-spectral               1
glg1-monte_carlo            1
glg1-lu                     1
results-insertion           1
results-log_processing      1
results-disk                1
results-flag                1
results-cws                 1
glg1-mndset                 1
results-network             1
results-merge               1
results-selection           1
results-roots               1
results-polling             1
results-pi                  1
results-sleep               1
results-short_circ_eval   

,Guideline,Benchmark,ART_ANOVA_p_holm,Median_Baseline_J,Median_Optimized_J,Conclusion_corrected
4,g1,results-lu,0.00000,6554.9112,1960.5226,Saves Energy
8,g1,results-sor,1.00000,503.2365,507.9336,No Statistically Significant Impact
10,g1,spectral_norm,0.04104,29.8125,32.0258,Wastes Energy
5,g1,results-monte_carlo,0.00000,2280.5289,2392.9026,Wastes Energy
7,g1,results-sieves-ros,1.00000,683.6377,685.0488,No Statistically Significant Impact
...,...,...,...,...,...,...
86,g28,results-lu,0.00000,6554.9112,1936.2477,Saves Energy
87,g28,results-monte_carlo,0.00480,2280.5289,2238.4877,Saves Energy
88,g28,results-sieves-ros,0.00000,683.6377,253.4102,Saves Energy
85,g28,nbody,0.00000,31.0222,34.0993,Wastes Energy


In [14]:
# Generate LaTeX table rows ready to paste 

def fmt_p(p):
    """Format p-values: very small ones as < 0.001, otherwise 3 decimals."""
    if p < 0.001:
        return "< 0.001"
    return f"{p:.3f}"

latex_lines = []
for _, row in table_df.iterrows():
    guideline = row['Guideline'].upper() 
    benchmark = row['Benchmark'].replace('_', '\\_')  
    p_holm = fmt_p(row['ART_ANOVA_p_holm'])
    med_base = f"{row['Median_Baseline_J']:.2f}"
    med_opt = f"{row['Median_Optimized_J']:.2f}"
    conclusion = row['Conclusion_corrected']

    line = f"{guideline} & {benchmark} & {p_holm} & {med_base} & {med_opt} & {conclusion} \\\\ \\hline"
    latex_lines.append(line)

latex_table_body = "\n".join(latex_lines)
print(latex_table_body)

with open("art_anova_energy_table_rows.tex", "w") as f:
    f.write(latex_table_body)
print("\n\nSaved to art_anova_energy_table_rows.tex")


G1 & results-lu & < 0.001 & 6554.91 & 1960.52 & Saves Energy \\ \hline
G1 & results-sor & 1.000 & 503.24 & 507.93 & No Statistically Significant Impact \\ \hline
G1 & spectral\_norm & 0.041 & 29.81 & 32.03 & Wastes Energy \\ \hline
G1 & results-monte\_carlo & < 0.001 & 2280.53 & 2392.90 & Wastes Energy \\ \hline
G1 & results-sieves-ros & 1.000 & 683.64 & 685.05 & No Statistically Significant Impact \\ \hline
G2 & results-sor & < 0.001 & 503.24 & 443.61 & Saves Energy \\ \hline
G2 & results-bubble-sort & < 0.001 & 450.63 & 265.00 & Saves Energy \\ \hline
G2 & results-insertion & < 0.001 & 65.30 & 3.31 & Saves Energy \\ \hline
G2 & results-merge & < 0.001 & 692.49 & 295.21 & Saves Energy \\ \hline
G2 & results-selection & 1.000 & 761.03 & 759.91 & No Statistically Significant Impact \\ \hline
G3 & results-lu & 1.000 & 6554.91 & 6588.19 & No Statistically Significant Impact \\ \hline
G3 & results-sor & 1.000 & 503.24 & 505.21 & No Statistically Significant Impact \\ \hline
G3 & spectral\_